In [ ]:
!unzip '/content/drive/MyDrive/files.zip'

Archive:  /content/drive/MyDrive/files.zip
replace wine_quality_nn.py? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

In [ ]:
# 1. Reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)


In [ ]:
# 2. Load data
df = pd.read_csv("WineQT.csv")
df = df.drop(columns=["Id"])   # Id is just a row number, not useful


In [ ]:
# 3. Turn quality score into 3 classes (Low/Medium/High)
def bucket_quality(q):
    if q <= 4:
        return 0   # Low
    elif q <= 6:
        return 1   # Medium
    else:
        return 2   # High

df["quality_class"] = df["quality"].apply(bucket_quality)
class_names = ["Low", "Medium", "High"]

print("Class distribution:")
print(df["quality_class"].value_counts().rename(index=dict(enumerate(class_names))))

In [ ]:
# 4. Features (X) and target (y)
X = df.drop(columns=["quality", "quality_class"])
y = df["quality_class"].values


In [ ]:

# 5. Train / validation split (80% train, 20% validation)
#    stratify=y keeps the same class ratio in both sets
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)


In [ ]:
# 6. Scale features
#    Neural networks train much better when all input columns
#    are on a similar scale (mean=0, std=1).
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

In [ ]:
# 7. Build the Neural Network
model = keras.Sequential([
    layers.Input(shape=(X_train_scaled.shape[1],)),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.3),          # prevents overfitting
    layers.Dense(32, activation="relu"),
    layers.Dropout(0.2),
    layers.Dense(3, activation="softmax")   # 3 output classes
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",  # correct loss for integer labels + multi-class
    metrics=["accuracy"]
)

model.summary()


In [ ]:
# 11. Save the trained model
model.save("wine_quality_model.keras")
print("Model saved to wine_quality_model.keras")

In [ ]:
# 9. Evaluate
val_loss, val_acc = model.evaluate(X_val_scaled, y_val, verbose=0)
print(f"\nFinal Validation Accuracy: {val_acc*100:.2f}%")
print(f"Final Validation Loss    : {val_loss:.4f}")

y_pred = np.argmax(model.predict(X_val_scaled, verbose=0), axis=1)
print("\nClassification Report:\n", classification_report(y_val, y_pred, target_names=class_names))


In [ ]:
# 10. Plot training curves + confusion matrix
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(history.history["accuracy"], label="Train")
axes[0].plot(history.history["val_accuracy"], label="Validation")
axes[0].set_title("Accuracy over Epochs")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Accuracy"); axes[0].legend()

axes[1].plot(history.history["loss"], label="Train")
axes[1].plot(history.history["val_loss"], label="Validation")
axes[1].set_title("Loss over Epochs")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Loss"); axes[1].legend()

cm = confusion_matrix(y_val, y_pred)
im = axes[2].imshow(cm, cmap="Blues")
axes[2].set_title("Confusion Matrix")
axes[2].set_xticks(range(3)); axes[2].set_xticklabels(class_names)
axes[2].set_yticks(range(3)); axes[2].set_yticklabels(class_names)
axes[2].set_xlabel("Predicted"); axes[2].set_ylabel("Actual")
for i in range(3):
    for j in range(3):
        axes[2].text(j, i, cm[i, j], ha="center", va="center",
                     color="white" if cm[i, j] > cm.max()/2 else "black")

plt.tight_layout()
plt.savefig("training_results.png", dpi=150)
print("\nSaved plots to training_results.png")

In [ ]:
# ---------------------------------------------------------
# 11. Save the trained model
# ---------------------------------------------------------
model.save("wine_quality_model.keras")
print("Model saved to wine_quality_model.keras")